### Day 8 Assignment: Governance, Unity Catalog, MERGE INTO/SCD & Pricing 


### Basic Tasks 


#### 1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table.

In [0]:
df = spark.read.csv("/Volumes/dev/demo/ex-volume/customers/customers_initial.csv", header=True, inferSchema=True)
df.write.mode("overwrite").saveAsTable("dev.demo.customers_initial")

In [0]:
df = spark.read.csv("/Volumes/dev/demo/ex-volume/customers/customers_changes.csv", header=True, inferSchema=True)
df.write.mode("overwrite").saveAsTable("dev.demo.customers_changes")

In [0]:
%sql
MERGE INTO dev.demo.customers_initial AS t
USING dev.demo.customers_changes AS s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN
  UPDATE SET t.name = s.name, t.email = s.email, t.city = s.city, t.state = s.state, t.signup_date = s.signup_date, t.phone = s.phone
WHEN NOT MATCHED
  THEN INSERT (customer_id, name, email, city, state, signup_date, phone) VALUES (s.customer_id, s.name, s.email, s.city, s.state, s.signup_date, s.phone)

#### 2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity Catalog permissions.

In [0]:
%sql
CREATE OR REPLACE VIEW dev.demo.customers_restricted AS
SELECT 
    customer_id,
    name,
    CONCAT(REPEAT("*", length(email)-4), RIGHT(email, 4)) AS email,
    city,
    state,
    signup_date,
    CONCAT(REPEAT("*", length(phone)-4), RIGHT(phone, 4)) AS phone
FROM
    dev.demo.customers_initial


In [0]:
%sql
GRANT USE CATALOG
ON CATALOG dev
TO `data_analysts`;

GRANT USE SCHEMA
ON SCHEMA dev.silver
TO `data_analysts`;

GRANT USE CATALOG
ON CATALOG dev
TO `data_engineers`;

GRANT USE SCHEMA
ON SCHEMA dev.silver
TO `data_engineers`;

In [0]:
%sql
GRANT SELECT 
ON VIEW dev.demo.customers_restricted
TO `data_analysts`;

GRANT SELECT 
ON VIEW dev.demo.customers_initial
TO `data_engineers`;

#### 3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms, what a DBU is billing for.

A DBU is Databricks' unit of compute consumption. The number of DBUs consumed depends on the compute resource and workload/runtime. DBUs are one component of the overall cost, alongside the underlying cloud infrastructure charges where applicable.

To check DBU consumption you need to go to Accounts and then Uses, there you can find the DBU uses and estimated cost across your Workspaces.

### Intermediate Tasks

#### 4. Build a full SCD Type 2 table

In [0]:
%sql
DROP TABLE IF EXISTS dev.silver.dim_customer_scd2;

CREATE TABLE dev.silver.dim_customer_scd2 (
    customer_id INT,
    name STRING,
    email STRING,
    city STRING,
    state STRING,
    signup_date DATE,
    phone BIGINT,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN
);

In [0]:
from pyspark.sql import functions as F

data = [
    (1, "John",  "john@gmail.com",  "Delhi",     "Delhi",      "2025-01-10", 9876543210),
    (2, "Alice", "alice@gmail.com", "Mumbai",    "Maharashtra","2025-02-15", 9876543211),
    (3, "Bob",   "bob@gmail.com",   "Chennai",   "Tamil Nadu", "2025-03-20", 9876543212),
    (4, "Sarah", "sarah@gmail.com", "Pune",      "Maharashtra","2025-04-10", 9876543213)
]

df = spark.createDataFrame(
    data,
    ["customer_id","name","email","city","state","signup_date","phone"]
).withColumn(
    "signup_date", F.to_date("signup_date")
)

df.createOrReplaceTempView("stg_customer")

In [0]:
%sql
--initial load
INSERT INTO dev.silver.dim_customer_scd2
SELECT
    customer_id,
    name,
    email,
    city,
    state,
    signup_date,
    phone,
    CURRENT_DATE AS start_date,
    DATE('9999-12-31') AS end_date,
    TRUE AS is_current
FROM stg_customer;

In [0]:
changed_data = [
    (1, "John",  "john@gmail.com",  "Bangalore", "Karnataka",  "2025-01-10", 9876543210),
    (2, "Alice", "alice@gmail.com", "Mumbai",    "Maharashtra","2025-02-15", 9876543211),
    (3, "Bob",   "bob@gmail.com",   "Chennai",   "Tamil Nadu", "2025-03-20", 9999999999),
    (5, "David", "david@gmail.com", "Hyderabad", "Telangana",   "2025-06-01", 9876543214)
]

changed_df = spark.createDataFrame(
    changed_data,
    ["customer_id","name","email","city","state","signup_date","phone"]
).withColumn(
    "signup_date", F.to_date("signup_date")
)

changed_df.createOrReplaceTempView("changed_customer")

In [0]:
%sql
MERGE INTO dev.silver.dim_customer_scd2 AS tgt

USING (
    SELECT
        src.*,
        CASE
            WHEN tgt.customer_id IS NULL THEN 'INSERT'
            WHEN NOT (
                src.name <=> tgt.name AND
                src.email <=> tgt.email AND
                src.city <=> tgt.city AND
                src.state <=> tgt.state AND
                src.phone <=> tgt.phone
            ) THEN 'UPDATE'
            ELSE 'NO_CHANGE'
        END AS action
    FROM changed_customer src
    LEFT JOIN dev.silver.dim_customer_scd2 tgt
        ON src.customer_id = tgt.customer_id
        AND tgt.is_current = TRUE
) src

ON tgt.customer_id = src.customer_id
AND tgt.is_current = TRUE
AND src.action = 'UPDATE'

WHEN MATCHED THEN
    UPDATE SET
        tgt.end_date = CURRENT_DATE,
        tgt.is_current = FALSE

WHEN NOT MATCHED AND src.action IN ('INSERT', 'UPDATE') THEN
    INSERT (
        customer_id,
        name,
        email,
        city,
        state,
        signup_date,
        phone,
        start_date,
        end_date,
        is_current
    )
    VALUES (
        src.customer_id,
        src.name,
        src.email,
        src.city,
        src.state,
        src.signup_date,
        src.phone,
        CURRENT_DATE,
        DATE('9999-12-31'),
        TRUE
    );

In [0]:
%sql
SELECT *
FROM dev.silver.dim_customer_scd2
ORDER BY customer_id, start_date;

#### 5. Point-in-Time Query

In [0]:
%sql
SELECT
    customer_id,
    name,
    city,
    state,
    start_date,
    end_date
FROM dev.silver.dim_customer_scd2
WHERE customer_id = 1
  AND DATE('2026-03-01') >= start_date
  AND DATE('2026-03-01') < end_date;

#### 6. DBU Cost Comparison

All-Purpose Compute is generally more expensive than Job Compute for scheduled workloads.

For Cyntexa, we should use **Job Compute** for the nightly pipeline because:

- **Lower Cost:** Avoids unnecessary compute charges.
- **No Idle Spend:** Compute terminates after the job finishes.
- **Reliability:** Fresh compute environment for each run.
- **Best Practice:** Designed for automated production pipelines.

**Action:** Configure the Databricks Workflow with **New Job Compute** instead of an existing All-Purpose cluster.

### Advanced Tasks 

#### 7. Governance Model

Cyntexa should classify **email, phone, and address** as sensitive customer PII.

**Access Model:**

- **Data Engineers:** Full access to Silver customer tables for pipeline processing.
- **Analysts:** Access only to approved Gold tables/views; PII should be masked.
- **Managers:** Access to aggregated business data without raw PII.
- **Restricted PII Access:** Only authorized users/groups should receive raw customer PII.

**Audit:** Unity Catalog audit/system tables should be used to review who accessed sensitive tables/views, when they accessed them, and what operation was performed.

**Principle:** Follow least-privilege access — users should receive only the data required for their role.

In [0]:
%sql
-- Create masked view
CREATE OR REPLACE VIEW dev.gold.customers_masked AS
SELECT
    customer_id,
    name,
    sha2(email, 256) AS email_masked,
    '**********' AS phone_masked,
    'RESTRICTED' AS address
FROM dev.silver.dim_customer_scd2
WHERE is_current = TRUE;

In [0]:
%sql
-- Grant permissions
GRANT SELECT ON VIEW dev.gold.customers_masked
TO `data_analysts`;

GRANT SELECT ON TABLE dev.silver.dim_customer_scd2
TO `data_engineers`;

In [0]:
%sql
-- Audit access
-- Filtering on map column by converting to string representation
SELECT
    event_time,
    user_identity,
    action_name,
    request_params
FROM system.access.audit
WHERE CAST(request_params AS STRING) LIKE '%dim_customer%'
ORDER BY event_time DESC;

#### 8. SCD2 — Multiple Column Changes

In [0]:
from pyspark.sql import functions as F

data = [
    (1, "John",  "john@gmail.com",  "Bangalore", "Karnataka", 9999999999),
    (2, "Alice", "alice@gmail.com", "Mumbai",    "Maharashtra", 8888888888),
    (3, "Bob",   "bob@gmail.com",   "Chennai",   "Tamil Nadu", 7777777777)
]

df = spark.createDataFrame(
    data,
    ["customer_id", "name", "email", "city", "state", "phone"]
)

df.createOrReplaceTempView("customer_changes")

In [0]:
%sql
MERGE INTO dev.silver.dim_customer_scd2 AS tgt

USING (
    SELECT
        src.*,
        CASE
            WHEN tgt.customer_id IS NULL THEN 'INSERT'

            WHEN src.name  IS DISTINCT FROM tgt.name
              OR src.email IS DISTINCT FROM tgt.email
              OR src.city  IS DISTINCT FROM tgt.city
              OR src.state IS DISTINCT FROM tgt.state
              OR src.phone IS DISTINCT FROM tgt.phone
            THEN 'UPDATE'

            ELSE 'NO_CHANGE'
        END AS action

    FROM customer_changes src

    LEFT JOIN dev.silver.dim_customer_scd2 tgt
        ON src.customer_id = tgt.customer_id
        AND tgt.is_current = TRUE
) src

ON tgt.customer_id = src.customer_id
AND tgt.is_current = TRUE
AND src.action = 'UPDATE'

WHEN MATCHED THEN
    UPDATE SET
        tgt.end_date = CURRENT_DATE,
        tgt.is_current = FALSE

WHEN NOT MATCHED
AND src.action IN ('INSERT', 'UPDATE')
THEN INSERT (
    customer_id,
    name,
    email,
    city,
    state,
    phone,
    start_date,
    end_date,
    is_current
)
VALUES (
    src.customer_id,
    src.name,
    src.email,
    src.city,
    src.state,
    src.phone,
    CURRENT_DATE,
    DATE('9999-12-31'),
    TRUE
);

#### 9. Customer Retention / Churn Report

SCD Type 2 history allows us to determine which customers were active at a specific point in time.

A customer is considered active when their SCD2 record is valid during the reporting period.

A current-state table would give the wrong historical answer because it only contains the customer's latest attributes. Once an address/status changes, the previous state is overwritten and we can no longer accurately determine what the customer looked like at an earlier point in time.

Therefore, SCD2 provides the historical validity required for accurate retention and churn analysis.

In [0]:
%sql
-- Example retention query
-- Assume we consider a customer retained if they were active in both January and March.
WITH customer_months AS (

    SELECT
        customer_id,

        CASE
            WHEN DATE('2026-01-31') >= start_date
             AND DATE('2026-01-31') < end_date
            THEN 1 ELSE 0
        END AS active_jan,

        CASE
            WHEN DATE('2026-03-31') >= start_date
             AND DATE('2026-03-31') < end_date
            THEN 1 ELSE 0
        END AS active_mar

    FROM dev.silver.dim_customer_scd2
),

customer_status AS (

    SELECT
        customer_id,
        MAX(active_jan) AS active_jan,
        MAX(active_mar) AS active_mar

    FROM customer_months
    GROUP BY customer_id
)

SELECT
    customer_id,
    active_jan,
    active_mar,

    CASE
        WHEN active_jan = 1 AND active_mar = 1
            THEN 'RETAINED'

        WHEN active_jan = 1 AND active_mar = 0
            THEN 'CHURNED'

        WHEN active_jan = 0 AND active_mar = 1
            THEN 'NEW/RETURNED'

        ELSE 'INACTIVE'
    END AS customer_status

FROM customer_status
ORDER BY customer_id;